# Assignment 1: Decoding States

---

## Task 5) Dual-Tone Multi-Frequency

[Dual-tone multi-frequency DTMF](https://en.wikipedia.org/wiki/Dual-tone_multi-frequency_signaling) signaling is an old way of transmitting dial pad keystrokes over the phone.
Each key/symbol is assigned a frequency pair: `[(1,697,1209), (2,697,1336), (3,697,1477), (A,697,1633), (4,770,1209), (5,770,1336), (6,770,1477), (B,770,1633), (7,852,1209), (8,852,1336), (9,852,1477), (C,852,1633), (*,941,1209), (0,941,1336), (#,941,1477), (D,941,1633)]`.
You can generate some DTMF sequences online, eg. <https://www.audiocheck.net/audiocheck_dtmf.php>

### Features

For feature computation, use librosa to compute the power spectrum (`librosa.stft` and `librosa.amplitude_to_db`), and extract the approx. band energy for each relevant frequency.

> Note: It's best to identify silence by the overall spectral energy and to normalize the band energies to sum up to one.

### Decoding

To decode DTMF sequences, we can use again dynamic programming, this time applied to states rather than edits.
For DTMF sequences, consider a small, fully connected graph that has 13 states: 0-9, A-D, \*, \# and _silence_.
As for the DP-matrix: the rows will denote the states and the columns represent the time; we will decode left-to-right (ie. time-synchronous), and at each time step, we will have to find the best step forward.
The main difference to edit distances or DTW is, that you may now also "go up" in the table, ie. change state freely.
For distance/similarity, use a template vector for each state that has `.5` for those two bins that need to be active.

When decoding a sequence, the idea is now that we remain in one state as long as the key is pressed; after that, the key may either be released (and the spectral energy is close to 0) hence we're in pause, or another key is pressed, hence it's a "direct" transition.
Thus, from the backtrack, collapse the sequence by digit and remove silence, eg. `44443-3-AAA` becomes `433A`.

---

### Preparation

In [1]:
import librosa
import numpy as np
from typing import List, Tuple

In [2]:
### Notice: librosa defaults to 22.050 Hz sample rate; adjust if needed!

DTMF_TONES = [
    ('1', 697, 1209), 
    ('2', 697, 1336), 
    ('3', 697, 1477), 
    ('A', 697, 1633),
    ('4', 770, 1209),
    ('5', 770, 1336),
    ('6', 770, 1477),
    ('B', 770, 1633),
    ('7', 852, 1209),
    ('8', 852, 1336),
    ('9', 852, 1477),
    ('C', 852, 1633),
    ('*', 941, 1209),
    ('0', 941, 1336),
    ('#', 941, 1477),
    ('D', 941, 1633)
]

### Implement the Decoding

In [3]:
### Notice: you will need a couple of helper functions...

def computeFrequencyMapping(sr: int, n_fft: int):
    # the frequency of the FFT
    fft_frequencies = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    relevantFrequencies = []
    for dtmf_tone in DTMF_TONES:
        relevantFrequencies.append(dtmf_tone[1])
        relevantFrequencies.append(dtmf_tone[2])

    # a map that maps the dmtf frequency to the closest frequency in
    # the FFT. The value is a tuple with the fft frequency and its index
    frequency_mapping = dict.fromkeys(relevantFrequencies, (0, 0))

    for dmtf_freq in frequency_mapping:
        for fft_freq in fft_frequencies:
            dist = abs(dmtf_freq - fft_freq)
            prev_dist = abs(dmtf_freq - frequency_mapping[dmtf_freq][0])
            if (dist < prev_dist):
                frequency_mapping[dmtf_freq] = (fft_freq, fft_frequencies.tolist().index(fft_freq))

    return frequency_mapping


def getFrequencies(bins: List[float], frequency_mapping) -> List[Tuple[int, float]]:
    return [(dtmf_freq, bins[frequency_mapping[dtmf_freq][1]]) for dtmf_freq in frequency_mapping.keys()]


def getDtmfTone(frequency_values: List[Tuple[int, float]]):
    # finds the dtmf tone for the two given frequencies
    def findDtmfTone(freq1: int, freq2: int) -> str:
        for dtmf_tone in DTMF_TONES:
            if (
                (dtmf_tone[1] == freq1 and dtmf_tone[2] == freq2) or
                (dtmf_tone[1] == freq2 and dtmf_tone[2] == freq1)
            ):
                return dtmf_tone[0]
        return "?"

    def isSilence():
        for v in frequency_values:
            if v[1] > -60:
                return False
        return True

    if isSilence():
        return " "

    frequency_values= sorted(frequency_values, key=lambda item: item[1])
    return findDtmfTone(frequency_values[-1][0], frequency_values[-2][0])


def decode(y: np.ndarray, sr: float) -> list:
    """
    Apply DTMF signal decoding.
    
    Arguments:
    y: Input signal.
    sr: Sample rate. 
    
    Returns list of DTMF-signals (no silence).
    """
    frequency_mapping = computeFrequencyMapping(sr=sr, n_fft=2048)

    S = np.abs(librosa.stft(y, n_fft=2048))
    D = np.transpose(librosa.amplitude_to_db(S, ref=np.max))

    tones = []
    for x in D:
        tone = getDtmfTone(getFrequencies(x, frequency_mapping))
        tones.append(tone)

    # collapse
    added = False
    collapsed = [tones[0]]
    prev_tone = tones[0]
    for tone in tones:
        if tone != prev_tone and tone != " ":
            collapsed.append(tone)
            prev_tone = tone
            added = True
        if added and tone == " ":
            prev_tone = " "
            added = False
    return collapsed

### Test the Decoding

In [4]:
SR = 22050
TEST_FILE = "data/audiocheck.net_dtmf_49_911_5880_0.wav"

y, sr = librosa.load(TEST_FILE, sr=SR)
print(decode(y=y, sr=sr))

['4', '9', '9', '1', '1', '5', '8', '8', '0', '0']
